# CodeDoctor: run in Colab

**File → Save a copy in Drive.** Run this notebook top to bottom inside Colab. These cells share one notebook environment; do not paste them into separate Python files.

For the VS Code + Streamlit copy-paste workshop, open [CodeDoctor.ipynb](https://github.com/RaghavD06/codedoctor-workshop/blob/main/CodeDoctor.ipynb) instead. That guide contains complete files with their imports.

Create your Gemini key privately at https://aistudio.google.com/apikey. Enter your key and the presenter's model ID in Stage 0. Never commit or screen-share the key.

Agent = LLM + Tools + Loop. An LLM is the model that decides which action to request.


## Stage 0: Setup


Install the Python packages and connect to Gemini using the OpenAI SDK, a library that sends requests to an AI service. `chat()` waits and retries when too many requests arrive at once.

> Never put your key in an environment variable for this workshop or share it on screen. Paste it privately, collapse this cell before presenting, and remove it before sharing your notebook. Presenter-only automated tests use an environment variable outside this notebook.

✅ Checkpoint: The cell finishes without an error. It does not call Gemini yet; Stage 1 checks the key and model ID.

In [ ]:
!pip install -q openai gradio pytest
import json, os, sys, time, shutil, subprocess
from pathlib import Path
from openai import OpenAI, RateLimitError

client = OpenAI(
    api_key="PASTE_YOUR_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "PASTE_CURRENT_FLASH_MODEL_ID"   # shown on the presenter's screen

def chat(**kwargs):
    """Calls the model, and waits + retries if we hit the free-tier rate limit."""
    for attempt in range(4):
        try:
            return client.chat.completions.create(model=MODEL, **kwargs)
        except RateLimitError:
            if attempt == 3:
                raise
            print("⏳ Rate limited, waiting 20 seconds...")
            time.sleep(20)


## The patient: buggy code


Meet the patient: a student grade report program. It has bugs. Don't fix them, that's CodeDoctor's job. A docstring is the description inside a function that tells us what it promises to do.

✅ Checkpoint: The broken program is printed, but has not been run.

In [ ]:
BUGGY_CODE = '''
def letter_grade(score):
    """Return the letter grade: 90+ is A, 80-89 is B, 70-79 is C, below 70 is F."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score > 70:
        return "C"
    else:
        return "F"


def class_report(students):
    """
    students: dict of name -> list of scores.
    Returns (report, topper):
      report: dict of name -> (average rounded to 1 decimal, letter grade).
              Students with no scores get (None, "N/A").
      topper: name of the student with the highest average.
    """
    report = {}
    for name, scores in students.items():
        avg = sum(scores) / len(score)
        report[name] = (round(avg, 1), letter_grade(avg))
    topper = max(report, key=lambda n: report[n][1])
    return report, topper


def class_average(students):
    """Return the average of every individual score in the class."""
    all_scores = [s for scores in students.values() for s in scores]
    return sum(all_scores) / len(students)


students = {
    "Asha": [92, 88, 95],
    "Ravi": [70, 72, 68],
    "Meera": [85, 79, 90],
    "Kabir": [],
}

report, topper = class_report(students)
for name, (avg, grade) in report.items():
    print(f"{name}: {avg} ({grade})")
print("Topper:", topper)
print("Class average:", round(class_average(students), 2))
'''
print(BUGGY_CODE)


## Stage 1: The brain in a jar


A plain LLM can suggest fixes, but it never runs anything here and cannot prove the answer works. Confident is not the same as correct.

✅ Checkpoint: It sounds confident. But did it run the code? Did it test anything? No.

In [ ]:
def ask(prompt):
    r = chat(messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print(ask("Fix all the bugs in this code and tell me if it works:\n" + BUGGY_CODE))


## Stage 2: Giving CodeDoctor hands


These are normal Python functions: the model reads their descriptions and ASKS us to run them. Code runs in a separate process with a timeout so an infinite loop cannot freeze this call. `pytest` runs test functions containing checks called assertions.

`tool_specs` describes each tool's name and inputs in JSON, a structured text format. `SAFE_ENV` limits inherited environment variables; a separate process and timeout are **not a security sandbox**. Run only trusted workshop examples in a disposable runtime.

✅ Checkpoint: The cell finishes quietly: the workspace and three tools are ready.

In [ ]:
WORKDIR = Path("/content/codedoctor_workspace")
SAFE_ENV = {"PATH": os.environ.get("PATH", ""), "HOME": "/tmp"}
# Windows needs SYSTEMROOT for Python networking dependencies used by pytest.
if os.name == "nt":
    SAFE_ENV["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", r"C:\Windows")

def reset_workspace():
    shutil.rmtree(WORKDIR, ignore_errors=True)
    WORKDIR.mkdir(parents=True)

def _run(cmd, timeout):
    try:
        p = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True,
                           timeout=timeout, env=SAFE_ENV)
        output = (p.stdout + p.stderr).strip() or "(no output)"
        status = "OK" if p.returncode == 0 else f"FAILED (exit code {p.returncode})"
        return f"{status}\n{output[-3000:]}"
    except subprocess.TimeoutExpired:
        return f"FAILED: timed out after {timeout}s (infinite loop?)"

def run_code(code):
    """Run a Python snippet in a separate process and return its output."""
    return _run([sys.executable, "-c", code], timeout=10)

def save_file(filename, content):
    """Save a file into the workspace (e.g. solution.py or test_solution.py)."""
    path = WORKDIR / Path(filename).name
    path.write_text(content)
    return f"Saved {path.name} ({len(content.splitlines())} lines)"

def run_tests(test_file=""):
    """Run pytest in the workspace (all tests, or one test file)."""
    target = [Path(test_file).name] if test_file else []
    return _run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                 "-p", "no:cacheprovider", *target], timeout=60)

def read_file(filename):
    path = WORKDIR / Path(filename).name
    return path.read_text() if path.exists() else ""

reset_workspace()
TOOLS = {"run_code": run_code, "save_file": save_file, "run_tests": run_tests}

tool_specs = [
    {"type": "function", "function": {
        "name": "run_code",
        "description": ("Run Python code in a fresh process and return its output and "
                        "any error. Files saved in the workspace (like solution.py) "
                        "can be imported."),
        "parameters": {"type": "object",
                       "properties": {"code": {"type": "string",
                                               "description": "Python code to run"}},
                       "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "save_file",
        "description": ("Save a file to the workspace. Use solution.py for the fixed "
                        "code and test_solution.py for pytest tests."),
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Full file contents"}},
                       "required": ["filename", "content"]}}},
    {"type": "function", "function": {
        "name": "run_tests",
        "description": "Run the pytest tests in the workspace and return the results.",
        "parameters": {"type": "object",
                       "properties": {"test_file": {"type": "string",
                                                    "description": "Optional: one test file to run. Leave empty to run all tests."}}}}},
]


## Stage 2: Try the tools by hand


We're doing manually what the agent will do automatically. Errors come back as TEXT instead of crashing this cell, so the agent can read and fix them.

✅ Checkpoint: You should see a NameError from the buggy code, then '1 passed' from the test.

In [ ]:
print(run_code(BUGGY_CODE))
print(save_file("hello.py", "def add(a, b):\n    return a + b\n"))
print(save_file("test_hello.py", "from hello import add\n\ndef test_add():\n    assert add(2, 3) == 5\n"))
print(run_tests())
reset_workspace()


## Stage 3: The agent loop


The loop repeats these steps:
- Send everything so far to the model.
- If it asks for a tool, run it and send back the result.
- Repeat with the new evidence.
- When it stops asking for tools, return its answer.

`max_steps` limits model turns so the loop cannot go on forever. A tool call is simply the model asking our Python program to run a named function.

✅ Checkpoint: The cell finishes quietly. You have defined the agent, and the next cell runs it.

In [ ]:
def run_agent(task, system_prompt, max_steps=15, log=print):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        msg = chat(messages=messages, tools=tool_specs).choices[0].message

        if not msg.tool_calls:                     # agent decided it's done
            return msg.content

        if msg.content:
            log(f"💭 {msg.content}")
        messages.append(msg.model_dump(exclude_none=True))

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            detail = f" → {args['filename']}" if "filename" in args else ""
            log(f"\n🔧 Step {step}: {call.function.name}{detail}")

            result = TOOLS[call.function.name](**args)
            icon = "❌" if result.startswith("FAILED") else "✅"
            log(f"{icon} {result[:600]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "Stopped: hit max steps without finishing."


## Stage 3: Run CodeDoctor


Give CodeDoctor its instructions and the patient. Watch the tool results: a red error is useful evidence, and passing tests are evidence only for the cases they actually check.

✅ Checkpoint: You should see ❌ errors, then fixes, then tests, ending in all tests passing. Runs vary: check the actual test output before trusting the report.

In [ ]:
DOCTOR_PROMPT = """You are CodeDoctor, an expert Python debugger. Follow this process:
1. Run the original code with run_code to see what actually happens.
2. Read every error carefully and find the root cause. Also look for LOGIC bugs:
   compare what the code does with what its docstrings promise.
3. Save the fully fixed code as solution.py using save_file. Put any demo/print
   code under if __name__ == "__main__": so tests can import it cleanly.
4. Write pytest tests in test_solution.py that check every promise in the
   docstrings, including edge cases. Import functions from solution.
5. Run run_tests. If a test fails, decide whether the code or the test is wrong,
   fix it, and run the tests again.
6. Never claim success until you have seen all tests pass.
Finish with a short report: a numbered list of each bug (what was wrong and how
you fixed it), then the final test result."""

def make_task(code):
    return ("Here is some broken Python code. Diagnose it, fix it, and prove "
            "the fix with tests.\n\n" + code)

reset_workspace()
report = run_agent(make_task(BUGGY_CODE), DOCTOR_PROMPT)
print("\n" + "=" * 60 + "\n🩺 CODEDOCTOR REPORT\n" + "=" * 60)
print(report)


## Stage 3: See the cure

Read the saved solution and tests, then run the fixed program yourself. Check that its output matches the promises in the docstrings.

✅ Checkpoint: Topper: Asha, and a class average of about 82.11 (it was 184.75 before, once the crash bugs were bypassed!).

> **Colab only:** this cell displays the results. There is no source file to fill for this step.

In [ ]:
print("===== FIXED CODE =====\n" + read_file("solution.py"))
print("===== TESTS IT WROTE =====\n" + read_file("test_solution.py"))
print("===== RUNNING THE FIXED PROGRAM =====")
print(run_code(read_file("solution.py")))


## Stage 4: Ship it, deploy as a web app


Gradio turns a Python function into a web app, and `share=True` gives it a public link that can open on a phone.

> **Public link:** submitted code runs on YOUR Colab machine using YOUR API key. Share only with trusted friends, use the optional password, and close the session afterward. The link stops working when your Colab session ends. The brief says links last up to 72 hours; the current [Gradio sharing guide](https://www.gradio.app/guides/sharing-your-app) says one week, while another Gradio guide still says 72 hours. Treat the link as temporary and check the launch message.

✅ Checkpoint: Open the gradio.live link on your phone. Paste your OWN trusted buggy code. Send it to a trusted friend.

In [ ]:
import gradio as gr

def codedoctor_app(buggy_code):
    reset_workspace()
    log_lines = []
    report = run_agent(make_task(buggy_code), DOCTOR_PROMPT, log=log_lines.append)
    return ("\n".join(log_lines), read_file("solution.py"),
            read_file("test_solution.py"), report)

demo = gr.Interface(
    fn=codedoctor_app,
    inputs=gr.Code(value=BUGGY_CODE, language="python",
                   label="🐛 Paste your broken Python code"),
    outputs=[
        gr.Textbox(label="🔧 What CodeDoctor did", lines=15),
        gr.Code(language="python", label="✅ Fixed code"),
        gr.Code(language="python", label="🧪 Tests it wrote"),
        gr.Markdown(label="🩺 Report"),
    ],
    title="🩺 CodeDoctor",
    description=("An AI agent that runs your code, finds the bugs, fixes them, "
                 "and proves the fix with tests."),
)
demo.launch(share=True)
# Want a password? Use: demo.launch(share=True, auth=("friend", "pick-a-password"))


## Stage 4 bonus: Your own bug

Choose one commented example and remove its leading `# ` characters. Each docstring describes the correct behavior, so the agent has something to test against.

✅ Checkpoint: Once you enable an example, its tests should expose the bug and verify the fix.

> **Colab/app practice only:** try these inputs in the running app. No project file needs this cell.

In [ ]:
# Uncomment ONE example below, then paste MY_CODE into the app.
# Or uncomment the final two lines to run it directly.

# MY_CODE = '''
# def reverse_text(text):
#     """Return all characters in reverse order; empty input returns an empty string."""
#     return text[:0:-1]
# '''

# MY_CODE = '''
# def fizzbuzz(n):
#     """Return strings for 1..n: Fizz for multiples of 3, Buzz for 5, FizzBuzz for both."""
#     result = []
#     for i in range(1, n + 1):
#         if i % 3 == 0:
#             result.append("Fizz")
#         elif i % 15 == 0:
#             result.append("FizzBuzz")
#         elif i % 5 == 0:
#             result.append("Buzz")
#         else:
#             result.append(str(i))
#     return result
# '''

# MY_CODE = '''
# def second_largest(numbers):
#     """Return the second-largest DISTINCT number; raise ValueError if fewer than two distinct values."""
#     if len(numbers) < 2:
#         raise ValueError("Need two distinct values")
#     return sorted(numbers)[-2]
# '''

# reset_workspace()
# print(run_agent(make_task(MY_CODE), DOCTOR_PROMPT))


## Wrap-up

**Agent = LLM + Tools + Loop**

**Code that runs is not the same as code that works.**

- Agent frameworks wrap this loop with reusable helpers.
- MCP (Model Context Protocol) is a standard way to plug tools into agents.
- Multi-agent means turning `run_agent` into a tool another agent calls.

Your fork includes the complete implementation. `app/gradio_app.py` provides Gradio; `app/app.py` provides Streamlit. Follow the README to deploy the Streamlit app with your own key and model in Secrets.

**You already know the core. Everything else is plumbing.**

# 🆘 Catch-up: run these if you fell behind

Choose **one** cell below. Each starts from scratch: paste your own key and the presenter's model ID into that cell before running it. These reset the workspace, so previous agent files are replaced.

A gets you ready for Stage 3. B runs through the cure. C also opens the web app.

## Catch-up A: Setup + patient + tools

Run this to catch up through Stage 2, including the tool check. You can then jump to the main Stage 3 loop cell.

✅ Checkpoint: NameError from the patient, then 1 passed; the workspace is reset for Stage 3.

In [ ]:
!pip install -q openai gradio pytest
import json, os, sys, time, shutil, subprocess
from pathlib import Path
from openai import OpenAI, RateLimitError

client = OpenAI(
    api_key="PASTE_YOUR_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "PASTE_CURRENT_FLASH_MODEL_ID"   # shown on the presenter's screen

def chat(**kwargs):
    """Calls the model, and waits + retries if we hit the free-tier rate limit."""
    for attempt in range(4):
        try:
            return client.chat.completions.create(model=MODEL, **kwargs)
        except RateLimitError:
            if attempt == 3:
                raise
            print("⏳ Rate limited, waiting 20 seconds...")
            time.sleep(20)


BUGGY_CODE = '''
def letter_grade(score):
    """Return the letter grade: 90+ is A, 80-89 is B, 70-79 is C, below 70 is F."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score > 70:
        return "C"
    else:
        return "F"


def class_report(students):
    """
    students: dict of name -> list of scores.
    Returns (report, topper):
      report: dict of name -> (average rounded to 1 decimal, letter grade).
              Students with no scores get (None, "N/A").
      topper: name of the student with the highest average.
    """
    report = {}
    for name, scores in students.items():
        avg = sum(scores) / len(score)
        report[name] = (round(avg, 1), letter_grade(avg))
    topper = max(report, key=lambda n: report[n][1])
    return report, topper


def class_average(students):
    """Return the average of every individual score in the class."""
    all_scores = [s for scores in students.values() for s in scores]
    return sum(all_scores) / len(students)


students = {
    "Asha": [92, 88, 95],
    "Ravi": [70, 72, 68],
    "Meera": [85, 79, 90],
    "Kabir": [],
}

report, topper = class_report(students)
for name, (avg, grade) in report.items():
    print(f"{name}: {avg} ({grade})")
print("Topper:", topper)
print("Class average:", round(class_average(students), 2))
'''
print(BUGGY_CODE)


def ask(prompt):
    r = chat(messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print(ask("Fix all the bugs in this code and tell me if it works:\n" + BUGGY_CODE))


WORKDIR = Path("/content/codedoctor_workspace")
SAFE_ENV = {"PATH": os.environ.get("PATH", ""), "HOME": "/tmp"}
# Windows needs SYSTEMROOT for Python networking dependencies used by pytest.
if os.name == "nt":
    SAFE_ENV["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", r"C:\Windows")

def reset_workspace():
    shutil.rmtree(WORKDIR, ignore_errors=True)
    WORKDIR.mkdir(parents=True)

def _run(cmd, timeout):
    try:
        p = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True,
                           timeout=timeout, env=SAFE_ENV)
        output = (p.stdout + p.stderr).strip() or "(no output)"
        status = "OK" if p.returncode == 0 else f"FAILED (exit code {p.returncode})"
        return f"{status}\n{output[-3000:]}"
    except subprocess.TimeoutExpired:
        return f"FAILED: timed out after {timeout}s (infinite loop?)"

def run_code(code):
    """Run a Python snippet in a separate process and return its output."""
    return _run([sys.executable, "-c", code], timeout=10)

def save_file(filename, content):
    """Save a file into the workspace (e.g. solution.py or test_solution.py)."""
    path = WORKDIR / Path(filename).name
    path.write_text(content)
    return f"Saved {path.name} ({len(content.splitlines())} lines)"

def run_tests(test_file=""):
    """Run pytest in the workspace (all tests, or one test file)."""
    target = [Path(test_file).name] if test_file else []
    return _run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                 "-p", "no:cacheprovider", *target], timeout=60)

def read_file(filename):
    path = WORKDIR / Path(filename).name
    return path.read_text() if path.exists() else ""

reset_workspace()
TOOLS = {"run_code": run_code, "save_file": save_file, "run_tests": run_tests}

tool_specs = [
    {"type": "function", "function": {
        "name": "run_code",
        "description": ("Run Python code in a fresh process and return its output and "
                        "any error. Files saved in the workspace (like solution.py) "
                        "can be imported."),
        "parameters": {"type": "object",
                       "properties": {"code": {"type": "string",
                                               "description": "Python code to run"}},
                       "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "save_file",
        "description": ("Save a file to the workspace. Use solution.py for the fixed "
                        "code and test_solution.py for pytest tests."),
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Full file contents"}},
                       "required": ["filename", "content"]}}},
    {"type": "function", "function": {
        "name": "run_tests",
        "description": "Run the pytest tests in the workspace and return the results.",
        "parameters": {"type": "object",
                       "properties": {"test_file": {"type": "string",
                                                    "description": "Optional: one test file to run. Leave empty to run all tests."}}}}},
]


print(run_code(BUGGY_CODE))
print(save_file("hello.py", "def add(a, b):\n    return a + b\n"))
print(save_file("test_hello.py", "from hello import add\n\ndef test_add():\n    assert add(2, 3) == 5\n"))
print(run_tests())
reset_workspace()


## Catch-up B: Through the agent

Run this to catch up through Stage 3, including the first LLM answer and the full agent run. It finishes by printing the solution, tests, and corrected program output.

✅ Checkpoint: Passing tests, a numbered bug report, Topper: Asha, and Class average: 82.11.

In [ ]:
!pip install -q openai gradio pytest
import json, os, sys, time, shutil, subprocess
from pathlib import Path
from openai import OpenAI, RateLimitError

client = OpenAI(
    api_key="PASTE_YOUR_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "PASTE_CURRENT_FLASH_MODEL_ID"   # shown on the presenter's screen

def chat(**kwargs):
    """Calls the model, and waits + retries if we hit the free-tier rate limit."""
    for attempt in range(4):
        try:
            return client.chat.completions.create(model=MODEL, **kwargs)
        except RateLimitError:
            if attempt == 3:
                raise
            print("⏳ Rate limited, waiting 20 seconds...")
            time.sleep(20)


BUGGY_CODE = '''
def letter_grade(score):
    """Return the letter grade: 90+ is A, 80-89 is B, 70-79 is C, below 70 is F."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score > 70:
        return "C"
    else:
        return "F"


def class_report(students):
    """
    students: dict of name -> list of scores.
    Returns (report, topper):
      report: dict of name -> (average rounded to 1 decimal, letter grade).
              Students with no scores get (None, "N/A").
      topper: name of the student with the highest average.
    """
    report = {}
    for name, scores in students.items():
        avg = sum(scores) / len(score)
        report[name] = (round(avg, 1), letter_grade(avg))
    topper = max(report, key=lambda n: report[n][1])
    return report, topper


def class_average(students):
    """Return the average of every individual score in the class."""
    all_scores = [s for scores in students.values() for s in scores]
    return sum(all_scores) / len(students)


students = {
    "Asha": [92, 88, 95],
    "Ravi": [70, 72, 68],
    "Meera": [85, 79, 90],
    "Kabir": [],
}

report, topper = class_report(students)
for name, (avg, grade) in report.items():
    print(f"{name}: {avg} ({grade})")
print("Topper:", topper)
print("Class average:", round(class_average(students), 2))
'''
print(BUGGY_CODE)


def ask(prompt):
    r = chat(messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print(ask("Fix all the bugs in this code and tell me if it works:\n" + BUGGY_CODE))


WORKDIR = Path("/content/codedoctor_workspace")
SAFE_ENV = {"PATH": os.environ.get("PATH", ""), "HOME": "/tmp"}
# Windows needs SYSTEMROOT for Python networking dependencies used by pytest.
if os.name == "nt":
    SAFE_ENV["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", r"C:\Windows")

def reset_workspace():
    shutil.rmtree(WORKDIR, ignore_errors=True)
    WORKDIR.mkdir(parents=True)

def _run(cmd, timeout):
    try:
        p = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True,
                           timeout=timeout, env=SAFE_ENV)
        output = (p.stdout + p.stderr).strip() or "(no output)"
        status = "OK" if p.returncode == 0 else f"FAILED (exit code {p.returncode})"
        return f"{status}\n{output[-3000:]}"
    except subprocess.TimeoutExpired:
        return f"FAILED: timed out after {timeout}s (infinite loop?)"

def run_code(code):
    """Run a Python snippet in a separate process and return its output."""
    return _run([sys.executable, "-c", code], timeout=10)

def save_file(filename, content):
    """Save a file into the workspace (e.g. solution.py or test_solution.py)."""
    path = WORKDIR / Path(filename).name
    path.write_text(content)
    return f"Saved {path.name} ({len(content.splitlines())} lines)"

def run_tests(test_file=""):
    """Run pytest in the workspace (all tests, or one test file)."""
    target = [Path(test_file).name] if test_file else []
    return _run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                 "-p", "no:cacheprovider", *target], timeout=60)

def read_file(filename):
    path = WORKDIR / Path(filename).name
    return path.read_text() if path.exists() else ""

reset_workspace()
TOOLS = {"run_code": run_code, "save_file": save_file, "run_tests": run_tests}

tool_specs = [
    {"type": "function", "function": {
        "name": "run_code",
        "description": ("Run Python code in a fresh process and return its output and "
                        "any error. Files saved in the workspace (like solution.py) "
                        "can be imported."),
        "parameters": {"type": "object",
                       "properties": {"code": {"type": "string",
                                               "description": "Python code to run"}},
                       "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "save_file",
        "description": ("Save a file to the workspace. Use solution.py for the fixed "
                        "code and test_solution.py for pytest tests."),
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Full file contents"}},
                       "required": ["filename", "content"]}}},
    {"type": "function", "function": {
        "name": "run_tests",
        "description": "Run the pytest tests in the workspace and return the results.",
        "parameters": {"type": "object",
                       "properties": {"test_file": {"type": "string",
                                                    "description": "Optional: one test file to run. Leave empty to run all tests."}}}}},
]


print(run_code(BUGGY_CODE))
print(save_file("hello.py", "def add(a, b):\n    return a + b\n"))
print(save_file("test_hello.py", "from hello import add\n\ndef test_add():\n    assert add(2, 3) == 5\n"))
print(run_tests())
reset_workspace()


def run_agent(task, system_prompt, max_steps=15, log=print):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        msg = chat(messages=messages, tools=tool_specs).choices[0].message

        if not msg.tool_calls:                     # agent decided it's done
            return msg.content

        if msg.content:
            log(f"💭 {msg.content}")
        messages.append(msg.model_dump(exclude_none=True))

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            detail = f" → {args['filename']}" if "filename" in args else ""
            log(f"\n🔧 Step {step}: {call.function.name}{detail}")

            result = TOOLS[call.function.name](**args)
            icon = "❌" if result.startswith("FAILED") else "✅"
            log(f"{icon} {result[:600]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "Stopped: hit max steps without finishing."


DOCTOR_PROMPT = """You are CodeDoctor, an expert Python debugger. Follow this process:
1. Run the original code with run_code to see what actually happens.
2. Read every error carefully and find the root cause. Also look for LOGIC bugs:
   compare what the code does with what its docstrings promise.
3. Save the fully fixed code as solution.py using save_file. Put any demo/print
   code under if __name__ == "__main__": so tests can import it cleanly.
4. Write pytest tests in test_solution.py that check every promise in the
   docstrings, including edge cases. Import functions from solution.
5. Run run_tests. If a test fails, decide whether the code or the test is wrong,
   fix it, and run the tests again.
6. Never claim success until you have seen all tests pass.
Finish with a short report: a numbered list of each bug (what was wrong and how
you fixed it), then the final test result."""

def make_task(code):
    return ("Here is some broken Python code. Diagnose it, fix it, and prove "
            "the fix with tests.\n\n" + code)

reset_workspace()
report = run_agent(make_task(BUGGY_CODE), DOCTOR_PROMPT)
print("\n" + "=" * 60 + "\n🩺 CODEDOCTOR REPORT\n" + "=" * 60)
print(report)


print("===== FIXED CODE =====\n" + read_file("solution.py"))
print("===== TESTS IT WROTE =====\n" + read_file("test_solution.py"))
print("===== RUNNING THE FIXED PROGRAM =====")
print(run_code(read_file("solution.py")))


## Catch-up C: Through deployment

Run this to catch up through Stage 4 from a fresh runtime. Keep the key private and allow a few minutes for the model calls before the public link appears.

✅ Checkpoint: The cure is printed and Gradio displays a public link.

In [ ]:
!pip install -q openai gradio pytest
import json, os, sys, time, shutil, subprocess
from pathlib import Path
from openai import OpenAI, RateLimitError

client = OpenAI(
    api_key="PASTE_YOUR_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "PASTE_CURRENT_FLASH_MODEL_ID"   # shown on the presenter's screen

def chat(**kwargs):
    """Calls the model, and waits + retries if we hit the free-tier rate limit."""
    for attempt in range(4):
        try:
            return client.chat.completions.create(model=MODEL, **kwargs)
        except RateLimitError:
            if attempt == 3:
                raise
            print("⏳ Rate limited, waiting 20 seconds...")
            time.sleep(20)


BUGGY_CODE = '''
def letter_grade(score):
    """Return the letter grade: 90+ is A, 80-89 is B, 70-79 is C, below 70 is F."""
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score > 70:
        return "C"
    else:
        return "F"


def class_report(students):
    """
    students: dict of name -> list of scores.
    Returns (report, topper):
      report: dict of name -> (average rounded to 1 decimal, letter grade).
              Students with no scores get (None, "N/A").
      topper: name of the student with the highest average.
    """
    report = {}
    for name, scores in students.items():
        avg = sum(scores) / len(score)
        report[name] = (round(avg, 1), letter_grade(avg))
    topper = max(report, key=lambda n: report[n][1])
    return report, topper


def class_average(students):
    """Return the average of every individual score in the class."""
    all_scores = [s for scores in students.values() for s in scores]
    return sum(all_scores) / len(students)


students = {
    "Asha": [92, 88, 95],
    "Ravi": [70, 72, 68],
    "Meera": [85, 79, 90],
    "Kabir": [],
}

report, topper = class_report(students)
for name, (avg, grade) in report.items():
    print(f"{name}: {avg} ({grade})")
print("Topper:", topper)
print("Class average:", round(class_average(students), 2))
'''
print(BUGGY_CODE)


def ask(prompt):
    r = chat(messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

print(ask("Fix all the bugs in this code and tell me if it works:\n" + BUGGY_CODE))


WORKDIR = Path("/content/codedoctor_workspace")
SAFE_ENV = {"PATH": os.environ.get("PATH", ""), "HOME": "/tmp"}
# Windows needs SYSTEMROOT for Python networking dependencies used by pytest.
if os.name == "nt":
    SAFE_ENV["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", r"C:\Windows")

def reset_workspace():
    shutil.rmtree(WORKDIR, ignore_errors=True)
    WORKDIR.mkdir(parents=True)

def _run(cmd, timeout):
    try:
        p = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True,
                           timeout=timeout, env=SAFE_ENV)
        output = (p.stdout + p.stderr).strip() or "(no output)"
        status = "OK" if p.returncode == 0 else f"FAILED (exit code {p.returncode})"
        return f"{status}\n{output[-3000:]}"
    except subprocess.TimeoutExpired:
        return f"FAILED: timed out after {timeout}s (infinite loop?)"

def run_code(code):
    """Run a Python snippet in a separate process and return its output."""
    return _run([sys.executable, "-c", code], timeout=10)

def save_file(filename, content):
    """Save a file into the workspace (e.g. solution.py or test_solution.py)."""
    path = WORKDIR / Path(filename).name
    path.write_text(content)
    return f"Saved {path.name} ({len(content.splitlines())} lines)"

def run_tests(test_file=""):
    """Run pytest in the workspace (all tests, or one test file)."""
    target = [Path(test_file).name] if test_file else []
    return _run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                 "-p", "no:cacheprovider", *target], timeout=60)

def read_file(filename):
    path = WORKDIR / Path(filename).name
    return path.read_text() if path.exists() else ""

reset_workspace()
TOOLS = {"run_code": run_code, "save_file": save_file, "run_tests": run_tests}

tool_specs = [
    {"type": "function", "function": {
        "name": "run_code",
        "description": ("Run Python code in a fresh process and return its output and "
                        "any error. Files saved in the workspace (like solution.py) "
                        "can be imported."),
        "parameters": {"type": "object",
                       "properties": {"code": {"type": "string",
                                               "description": "Python code to run"}},
                       "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "save_file",
        "description": ("Save a file to the workspace. Use solution.py for the fixed "
                        "code and test_solution.py for pytest tests."),
        "parameters": {"type": "object",
                       "properties": {"filename": {"type": "string"},
                                      "content": {"type": "string",
                                                  "description": "Full file contents"}},
                       "required": ["filename", "content"]}}},
    {"type": "function", "function": {
        "name": "run_tests",
        "description": "Run the pytest tests in the workspace and return the results.",
        "parameters": {"type": "object",
                       "properties": {"test_file": {"type": "string",
                                                    "description": "Optional: one test file to run. Leave empty to run all tests."}}}}},
]


print(run_code(BUGGY_CODE))
print(save_file("hello.py", "def add(a, b):\n    return a + b\n"))
print(save_file("test_hello.py", "from hello import add\n\ndef test_add():\n    assert add(2, 3) == 5\n"))
print(run_tests())
reset_workspace()


def run_agent(task, system_prompt, max_steps=15, log=print):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        msg = chat(messages=messages, tools=tool_specs).choices[0].message

        if not msg.tool_calls:                     # agent decided it's done
            return msg.content

        if msg.content:
            log(f"💭 {msg.content}")
        messages.append(msg.model_dump(exclude_none=True))

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            detail = f" → {args['filename']}" if "filename" in args else ""
            log(f"\n🔧 Step {step}: {call.function.name}{detail}")

            result = TOOLS[call.function.name](**args)
            icon = "❌" if result.startswith("FAILED") else "✅"
            log(f"{icon} {result[:600]}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    return "Stopped: hit max steps without finishing."


DOCTOR_PROMPT = """You are CodeDoctor, an expert Python debugger. Follow this process:
1. Run the original code with run_code to see what actually happens.
2. Read every error carefully and find the root cause. Also look for LOGIC bugs:
   compare what the code does with what its docstrings promise.
3. Save the fully fixed code as solution.py using save_file. Put any demo/print
   code under if __name__ == "__main__": so tests can import it cleanly.
4. Write pytest tests in test_solution.py that check every promise in the
   docstrings, including edge cases. Import functions from solution.
5. Run run_tests. If a test fails, decide whether the code or the test is wrong,
   fix it, and run the tests again.
6. Never claim success until you have seen all tests pass.
Finish with a short report: a numbered list of each bug (what was wrong and how
you fixed it), then the final test result."""

def make_task(code):
    return ("Here is some broken Python code. Diagnose it, fix it, and prove "
            "the fix with tests.\n\n" + code)

reset_workspace()
report = run_agent(make_task(BUGGY_CODE), DOCTOR_PROMPT)
print("\n" + "=" * 60 + "\n🩺 CODEDOCTOR REPORT\n" + "=" * 60)
print(report)


print("===== FIXED CODE =====\n" + read_file("solution.py"))
print("===== TESTS IT WROTE =====\n" + read_file("test_solution.py"))
print("===== RUNNING THE FIXED PROGRAM =====")
print(run_code(read_file("solution.py")))


import gradio as gr

def codedoctor_app(buggy_code):
    reset_workspace()
    log_lines = []
    report = run_agent(make_task(buggy_code), DOCTOR_PROMPT, log=log_lines.append)
    return ("\n".join(log_lines), read_file("solution.py"),
            read_file("test_solution.py"), report)

demo = gr.Interface(
    fn=codedoctor_app,
    inputs=gr.Code(value=BUGGY_CODE, language="python",
                   label="🐛 Paste your broken Python code"),
    outputs=[
        gr.Textbox(label="🔧 What CodeDoctor did", lines=15),
        gr.Code(language="python", label="✅ Fixed code"),
        gr.Code(language="python", label="🧪 Tests it wrote"),
        gr.Markdown(label="🩺 Report"),
    ],
    title="🩺 CodeDoctor",
    description=("An AI agent that runs your code, finds the bugs, fixes them, "
                 "and proves the fix with tests."),
)
demo.launch(share=True)
# Want a password? Use: demo.launch(share=True, auth=("friend", "pick-a-password"))


## Troubleshooting

| Problem | Fix |
|---|---|
| 401 / “API key not valid” | Re-copy the key from AI Studio and check for spaces. |
| 404 / “model not found” | Use the model ID on the presenter's screen. |
| 429 / rate limit | The code waits automatically. If it keeps failing, wait a minute. |
| NameError in a notebook cell | An earlier cell wasn't run: Runtime → Run all. The patient's first NameError is intentional. |
| Agent says “Stopped: hit max steps” | Rerun the cell. Agents aren't deterministic: identical inputs can produce different runs. |
| Gradio link not showing | Rerun the deploy cell and wait 10–20 seconds. |
| Colab won't open | Use a personal Gmail account, not a college/work account. |